# NIPOPETO — kiểm thử tải Pha 1 trên Kaggle CPU

Tải file `.ipynb` này lên Kaggle Notebook mới, chọn **Accelerator: None** và bật **Internet** trong Session options. Chạy các cell từ trên xuống. Backend, PostgreSQL và Locust dùng chung CPU Kaggle; không cần Docker hay tài khoản database của máy cá nhân.

Đợi nhóm commit/push bản chốt, điền **full SHA 40 ký tự** ở cell tiếp theo. Mỗi lượt tải tạo database PostgreSQL tạm riêng. Khi xong, tải ZIP trong mục **Output** ở bên phải (bấm refresh nếu chưa thấy), rồi gửi ZIP để đối chiếu. Nếu clone/cài đặt lỗi, xem output ngay dưới cell lỗi; nếu một run lỗi, xem `manifest.json` và `setup.log`/`api.log`/`load.log` trong ZIP.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, re, shutil, subprocess, urllib.request, zipfile

REPO_URL = "https://github.com/Nguyenquanghong/Pet-healthcare.git"
COMMIT_SHA = "DIEN_FULL_SHA_40_KY_TU_SAU_KHI_PUSH"
RUN_LABEL = "B1"  # Đổi thành B2 cho bản cải tiến Pha 2.
WORKING = Path("/kaggle/working")

assert Path("/kaggle").exists() and WORKING.exists(), "Notebook này phải chạy trong Kaggle."
assert re.fullmatch(r"[0-9a-fA-F]{40}", COMMIT_SHA), "Điền full COMMIT_SHA 40 ký tự của bản đã push."
assert REPO_URL.startswith("https://github.com/") and REPO_URL.endswith(".git"), "Điền URL GitHub public dạng https://github.com/...git"
assert re.fullmatch(r"[A-Za-z0-9_-]+", RUN_LABEL), "RUN_LABEL chỉ gồm chữ, số, _ hoặc -."
repo = WORKING / "Pet-healthcare"
print("Run label:", RUN_LABEL, "| CPU logic:", os.cpu_count(), "| Source:", COMMIT_SHA)


## Lấy đúng source

Cell này dùng lại checkout sạch nếu chạy lại với cùng SHA. Nó dừng khi thư mục checkout có thay đổi hoặc khác SHA, để tránh gắn sai nhãn kết quả. Không tự xóa checkout cũ.


In [ ]:
if not repo.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()
dirty = subprocess.check_output(["git", "status", "--porcelain"], cwd=repo, text=True).strip()
if actual.lower() != COMMIT_SHA.lower():
    assert not dirty, "Checkout hiện có thay đổi; hãy kiểm tra trước khi đổi commit."
    subprocess.run(["git", "fetch", "origin", COMMIT_SHA], cwd=repo, check=True)
    subprocess.run(["git", "checkout", "--detach", COMMIT_SHA], cwd=repo, check=True)
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()
dirty = subprocess.check_output(["git", "status", "--porcelain"], cwd=repo, text=True).strip()
assert actual.lower() == COMMIT_SHA.lower() and not dirty, "SHA không khớp hoặc working tree không sạch."
print("Source commit:", actual, "| dirty_worktree:", bool(dirty))


## Cài công cụ

Kaggle cần Internet để tải PostgreSQL, Node.js 24 và các gói Python nếu chưa có sẵn. Lỗi cài đặt sẽ hiện ở output của cell này. PostgreSQL được runner khởi tạo trong thư mục tạm cho **từng** lượt đo.


In [ ]:
if not shutil.which("pg_config") or not shutil.which("createdb"):
    assert os.geteuid() == 0, "Không có quyền cài PostgreSQL trong phiên này."
    subprocess.run(["apt-get", "update"], check=True)
    subprocess.run(["apt-get", "install", "-y", "postgresql", "postgresql-client", "libpq-dev"], check=True)
node_dir = WORKING / "node24"
if node_dir.exists() and (node_dir / "bin/node").exists():
    os.environ["PATH"] = str(node_dir / "bin") + os.pathsep + os.environ["PATH"]
node_ok = shutil.which("node") and subprocess.check_output(["node", "--version"], text=True).strip().startswith("v24.")
if not node_ok:
    version = "v24.15.0"
    name = f"node-{version}-linux-x64.tar.xz"
    base = f"https://nodejs.org/dist/{version}/"
    archive = WORKING / name
    urllib.request.urlretrieve(base + name, archive)
    checksums = urllib.request.urlopen(base + "SHASUMS256.txt").read().decode()
    expected = next(line.split()[0] for line in checksums.splitlines() if line.split()[-1] == name)
    assert hashlib.sha256(archive.read_bytes()).hexdigest() == expected, "Node checksum không khớp."
    node_dir.mkdir(exist_ok=True)
    subprocess.run(["tar", "-xJf", str(archive), "-C", str(node_dir), "--strip-components=1"], check=True)
    os.environ["PATH"] = str(node_dir / "bin") + os.pathsep + os.environ["PATH"]
subprocess.run(["python", "-m", "pip", "install", "-r", str(repo / "benchmarks/kaggle/requirements.txt")], check=True)
for program, args in {"node": ["--version"], "npm": ["--version"], "pg_config": ["--version"], "locust": ["--version"]}.items():
    print(program, subprocess.check_output([program, *args], text=True).strip())


## Chạy ma trận

Hai profile × 1/10/50/100 người dùng × 2 lần = tối đa 16 run. Mỗi run warmup 15 giây, đo 60 giây và tăng 5 người dùng/giây. Có think time 0,5–1,5 giây; 60 giây gồm cả ramp-up. Dừng khi run lỗi, error rate >1% hoặc p95 >2.000 ms. Nếu chạy lại cell, hệ thống tạo thư mục mới và giữ bằng chứng cũ.


In [ ]:
evidence = WORKING / f"nipopeto-evidence-{RUN_LABEL}-{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}"
evidence.mkdir(exist_ok=False)
run_error = None
run_count = 0
for profile in ("ReadHeavyUser", "MixedUser"):
    if run_error:
        break
    for users in (1, 10, 50, 100):
        if run_error:
            break
        for repeat in (1, 2):
            folder = evidence / f"{RUN_LABEL}_{profile}_{users}_{repeat}"
            args = ["python", str(repo / "benchmarks/kaggle/runner.py"), "--repo", str(repo), "--output", str(folder), "--profile", profile, "--users", str(users), "--spawn-rate", "5", "--warmup", "15s", "--duration", "60s", "--seed", "phase1"]
            print("Running", folder.name, flush=True)
            result = subprocess.run(args, cwd=repo, env=os.environ.copy())
            run_count += 1
            if result.returncode:
                run_error = f"{folder.name}: exit {result.returncode}. Xem manifest và log trong ZIP."
                break
print("Đã thử:", run_count, "/16. Bằng chứng:", evidence)
if run_error:
    print("Dừng theo guardrail:", run_error)
(evidence / "notebook-summary.json").write_text(json.dumps({
    "commit": actual, "label": RUN_LABEL, "attempted_runs": run_count,
    "expected_runs": 16, "stop_reason": run_error,
}, ensure_ascii=False, indent=2), encoding="utf-8")


## Tổng hợp và tải kết quả

Tổng hợp từ manifest, không gộp warmup. `requests` trong manifest lấy từ CSV snapshot cuối và có thể khác tổng ở cuối `load.log`; cần đối chiếu khi lập báo cáo. ZIP chỉ chứa manifest, CSV, thống kê tài nguyên và log, không gồm source, database tạm hay biến môi trường. Nếu không thấy ZIP trong Output, bấm refresh.


In [ ]:
rows = []
allowed = {"manifest.json", "load_stats.csv", "load_failures.csv", "load_stats_history.csv", "resources.json", "setup.log", "api.log", "warmup.log", "load.log", "postgres.log"}
for folder in sorted(evidence.iterdir()):
    if not folder.is_dir():
        continue
    manifest = folder / "manifest.json"
    data = json.loads(manifest.read_text()) if manifest.exists() else {}
    results = data.get("results") or {}
    rows.append({"run": folder.name, "status": data.get("status", "NO_MANIFEST"), "requests": results.get("requests"), "failures": results.get("failures"), "rps": results.get("achieved_rps"), "p95_ms": results.get("p95_ms"), "threshold_exceeded": data.get("threshold_exceeded")})
for row in rows:
    print(row)
print("Completed:", sum(row["status"] == "completed" for row in rows), "/16; error:", run_error or "không")
archive = evidence.with_suffix(".zip")
assert not archive.exists(), "ZIP đã tồn tại; hãy chọn RUN_LABEL khác hoặc kiểm tra thư mục cũ."
with zipfile.ZipFile(archive, "x", zipfile.ZIP_DEFLATED) as output:
    summary = evidence / "notebook-summary.json"
    if summary.exists():
        output.write(summary, f"{evidence.name}/{summary.name}")
    for folder in sorted(evidence.iterdir()):
        if folder.is_dir():
            for item in folder.iterdir():
                if item.is_file() and item.name in allowed:
                    member = f"{evidence.name}/{folder.name}/{item.name}"
                    if item.suffix == ".log":
                        safe = item.read_text(encoding="utf-8", errors="replace")
                        safe = re.sub(r"postgres(?:ql)?://[^\s'\"]+", "[REDACTED_DATABASE_URL]", safe, flags=re.IGNORECASE)
                        safe = re.sub(r"Bearer\s+[A-Za-z0-9._~-]+", "Bearer [REDACTED]", safe, flags=re.IGNORECASE)
                        output.writestr(member, safe)
                    else:
                        output.write(item, member)
assert archive.exists() and archive.stat().st_size > 0
print("Tải ZIP từ mục Output:", archive, "| bytes:", archive.stat().st_size)
